# A circuit lookup table: any Boolean function as a memory read

[`memory_address_decoder_boolean_to_physics.ipynb`](memory_address_decoder_boolean_to_physics.ipynb)
built a Boolean AND-gate decoder that selects ONE physical DRAM cell out of
a grid, given a `(row, col)` address. An FPGA lookup table (LUT) reuses the
exact same one-hot minterm decoder — [`dgs/memory_address_decoder.py`](../dgs/memory_address_decoder.py)'s
`decoder_outputs` — but instead of selecting a *stored data* cell, it selects
one bit of a *stored truth table*. That's the entire trick behind how modern
FPGAs implement arbitrary logic: bake a function's truth table into $2^n$
memory bits once, then "evaluate" the function forever after by nothing
more than an address decode + a big OR gate.

[`dgs/lut_circuit.py`](../dgs/lut_circuit.py) builds that LUT read circuit
from scratch, verifies it reproduces real functions from
[`dgs/computer_engineering.py`](../dgs/computer_engineering.py) exactly, and
computes the actual engineering tradeoff: a LUT always costs a *fixed*
$2^n$ bits, while a minimized gate-level implementation
([`dgs/boolean_algebra.py`](../dgs/boolean_algebra.py)'s Quine–McCluskey
`minimize_sop`) costs less for "nice" functions — and, for genuinely
incompressible functions like XOR/parity, costs *more*.


In [1]:
import sys, pathlib

REPO = pathlib.Path(r"D:/Summer2026/Dispersion-Assisted-GS-Phase-Recovery")
sys.path.insert(0, str(REPO))
from dgs import lut_circuit as lc
from dgs import memory_address_decoder as mad
from dgs.boolean_algebra import TruthTable, minimize_sop
from dgs.computer_engineering import full_adder

checks = []


def check(label, condition):
    checks.append((label, bool(condition)))
    print(f"{'PASS' if condition else 'FAIL'}  —  {label}")


## 1. Bake a function into LUT bits, then read it back with pure circuit logic

`synthesize_lut` uses `dgs.boolean_algebra.TruthTable`'s own enumeration to
produce the $2^n$ stored bits. `lut_read` then evaluates the function for a
given input using **only** `memory_address_decoder.decoder_outputs`
(the same one-hot AND-gate decoder from the memory notebook) plus an
AND/OR — no gate anywhere computes majority or XOR at runtime.


In [2]:
majority3 = lambda A, B, C: (A & B) | (B & C) | (A & C)
xor3 = lambda A, B, C: A ^ B ^ C

lut_bits, tt = lc.synthesize_lut(3, majority3)
print("majority-3 truth table, in minterm-address order:")
tt.show()
print("\nLUT content (address j -> stored bit):", lut_bits)

check("LUT content matches majority-3's minterms {3,5,6,7}", lut_bits == (0, 0, 0, 1, 0, 1, 1, 1))


majority-3 truth table, in minterm-address order:
A  B  C  |  F
-------------
0   0   0  |  0
0   0   1  |  0
0   1   0  |  0
0   1   1  |  1
1   0   0  |  0
1   0   1  |  1
1   1   0  |  1
1   1   1  |  1

LUT content (address j -> stored bit): (0, 0, 0, 1, 0, 1, 1, 1)
PASS  —  LUT content matches majority-3's minterms {3,5,6,7}


In [3]:
print("Reading the LUT circuit for every input (decode + AND + OR):\n")
for bits, expected in tt.rows:
    got = lc.lut_read(lut_bits, bits)
    print(f"  input={bits}  ->  LUT circuit output={got}  (function says {expected})")

check("LUT circuit reproduces majority-3 for all 8 inputs (exhaustive)", lc.verify_lut(3, majority3))
check("LUT circuit reproduces XOR-3 for all 8 inputs (exhaustive)", lc.verify_lut(3, xor3))
check("LUT circuit reproduces XOR-4 for all 16 inputs (exhaustive)",
      lc.verify_lut(4, lambda A, B, C, D: A ^ B ^ C ^ D))


Reading the LUT circuit for every input (decode + AND + OR):

  input=(0, 0, 0)  ->  LUT circuit output=0  (function says 0)
  input=(0, 0, 1)  ->  LUT circuit output=0  (function says 0)
  input=(0, 1, 0)  ->  LUT circuit output=0  (function says 0)
  input=(0, 1, 1)  ->  LUT circuit output=1  (function says 1)
  input=(1, 0, 0)  ->  LUT circuit output=0  (function says 0)
  input=(1, 0, 1)  ->  LUT circuit output=1  (function says 1)
  input=(1, 1, 0)  ->  LUT circuit output=1  (function says 1)
  input=(1, 1, 1)  ->  LUT circuit output=1  (function says 1)
PASS  —  LUT circuit reproduces majority-3 for all 8 inputs (exhaustive)
PASS  —  LUT circuit reproduces XOR-3 for all 8 inputs (exhaustive)
PASS  —  LUT circuit reproduces XOR-4 for all 16 inputs (exhaustive)


## 2. Cross-check against a REAL circuit: `full_adder`'s own Cout and Sum

`dgs.computer_engineering.full_adder`'s docstring already states
`Cout = majority(A,B,Cin)` and `S = XOR(XOR(A,B),Cin)`. Confirm that
directly, then confirm the LUT built from `majority3` reproduces
`full_adder`'s actual `Cout` output for every input — the LUT isn't being
tested against a toy function, it's being tested against the same adder
this repo's own ripple-carry-adder code depends on.


In [4]:
all_match = True
for A in (0, 1):
    for B in (0, 1):
        for Cin in (0, 1):
            fa = full_adder(A, B, Cin)
            cout_via_lut = lc.lut_read(lut_bits, (A, B, Cin))
            print(f"  A={A} B={B} Cin={Cin}:  full_adder Cout={fa['Cout']}  LUT(majority3) output={cout_via_lut}")
            all_match &= (fa["Cout"] == cout_via_lut == majority3(A, B, Cin))

check("LUT(majority3) reproduces full_adder's actual Cout for every input", all_match)


  A=0 B=0 Cin=0:  full_adder Cout=0  LUT(majority3) output=0
  A=0 B=0 Cin=1:  full_adder Cout=0  LUT(majority3) output=0
  A=0 B=1 Cin=0:  full_adder Cout=0  LUT(majority3) output=0
  A=0 B=1 Cin=1:  full_adder Cout=1  LUT(majority3) output=1
  A=1 B=0 Cin=0:  full_adder Cout=0  LUT(majority3) output=0
  A=1 B=0 Cin=1:  full_adder Cout=1  LUT(majority3) output=1
  A=1 B=1 Cin=0:  full_adder Cout=1  LUT(majority3) output=1
  A=1 B=1 Cin=1:  full_adder Cout=1  LUT(majority3) output=1
PASS  —  LUT(majority3) reproduces full_adder's actual Cout for every input


## 3. The digital-circuit ratio: LUT bits per minimized literal

A LUT costs $2^n$ bits, full stop — that number doesn't care whether the
function is trivial or maximally complex. A minimized SOP does care. The
ratio `lut_bits / literal_count` says which side of that tradeoff a given
function falls on:

- **ratio > 1** — the LUT is *wasteful*: a minimized gate circuit would be
  cheaper (majority-3 compresses to `BC + AC + AB`, just 6 literals).
- **ratio < 1** — the LUT is *cheaper than gates*: the function doesn't
  compress at all. XOR/parity is the textbook case — on a K-map, no two
  adjacent cells ever agree, so **every minterm is its own prime
  implicant** and the "minimized" SOP is just the full sum of minterms.


In [5]:
examples = [
    ("majority-3  (full_adder Cout)", 3, majority3),
    ("XOR-3  (full_adder Sum)", 3, xor3),
    ("XOR-4", 4, lambda A, B, C, D: A ^ B ^ C ^ D),
]

for name, n, fn in examples:
    stats = lc.lut_vs_minimized_gates_ratio(n, fn)
    print(f"{name}")
    print(f"  minimized SOP: {stats['sop']}")
    print(f"  literals={stats['literal_count']:3d}   LUT bits={stats['lut_bits']:3d}   "
          f"ratio={stats['ratio_lut_bits_per_literal']:.3f}\n")

maj_stats = lc.lut_vs_minimized_gates_ratio(3, majority3)
xor3_stats = lc.lut_vs_minimized_gates_ratio(3, xor3)
xor4_stats = lc.lut_vs_minimized_gates_ratio(4, lambda A, B, C, D: A ^ B ^ C ^ D)

check("Majority-3 compresses well: LUT uses MORE raw bits than minimized gates (ratio > 1)",
      maj_stats["ratio_lut_bits_per_literal"] > 1.0)
check("XOR-3 does not compress: LUT is actually CHEAPER than minimized gates (ratio < 1)",
      xor3_stats["ratio_lut_bits_per_literal"] < 1.0)
check("XOR's non-compressibility gets WORSE as n grows (ratio shrinks further below 1)",
      xor4_stats["ratio_lut_bits_per_literal"] < xor3_stats["ratio_lut_bits_per_literal"])


majority-3  (full_adder Cout)
  minimized SOP: F = BC + AC + AB
  literals=  6   LUT bits=  8   ratio=1.333

XOR-3  (full_adder Sum)
  minimized SOP: F = ~A~BC + ~AB~C + A~B~C + ABC
  literals= 12   LUT bits=  8   ratio=0.667

XOR-4
  minimized SOP: F = ~A~B~CD + ~A~BC~D + ~AB~C~D + ~ABCD + A~B~C~D + A~BCD + AB~CD + ABC~D
  literals= 32   LUT bits= 16   ratio=0.500

PASS  —  Majority-3 compresses well: LUT uses MORE raw bits than minimized gates (ratio > 1)
PASS  —  XOR-3 does not compress: LUT is actually CHEAPER than minimized gates (ratio < 1)
PASS  —  XOR's non-compressibility gets WORSE as n grows (ratio shrinks further below 1)


This is exactly why real FPGAs standardize on small LUTs (4–6 input,
not 20-input): for arbitrary logic, the $2^n$ storage cost is fixed and
predictable, and functions like XOR/parity — which show up constantly in
real designs (checksums, ECC, CRCs) — get *no benefit at all* from
gate-level minimization. A LUT trades away the occasional win on
"nice" functions for a flat, guaranteed-worst-case cost on the functions
that would otherwise blow up.

## Final grade

In [6]:
failures = [label for label, ok in checks if not ok]
print(f"{len(checks) - len(failures)}/{len(checks)} checks passed")

if failures:
    raise AssertionError("Failed checks: " + ", ".join(failures))
else:
    print("\nALL CHECKS PASSED — the LUT circuit (built from the same one-hot decoder as the "
          "memory address notebook) exactly reproduces majority-3, XOR-3/4, and the real "
          "full_adder's Cout, and the LUT-vs-gates ratio behaves exactly as digital-design "
          "theory predicts for compressible vs. incompressible functions.")


8/8 checks passed

ALL CHECKS PASSED — the LUT circuit (built from the same one-hot decoder as the memory address notebook) exactly reproduces majority-3, XOR-3/4, and the real full_adder's Cout, and the LUT-vs-gates ratio behaves exactly as digital-design theory predicts for compressible vs. incompressible functions.
